**Kaggle Kernel Config:**

In [ ]:
# KAGGLE_CONFIG: execute = true
# KAGGLE_CONFIG: slug = "notebook-consuming-tpu"
# KAGGLE_CONFIG: language = "python"
# KAGGLE_CONFIG: kernel_type = "notebook"
# KAGGLE_CONFIG: is_private = false
# KAGGLE_CONFIG: enable_gpu = false
# KAGGLE_CONFIG: enable_tpu = true
# KAGGLE_CONFIG: enable_internet = false
# KAGGLE_CONFIG: accelerator = "Tpu1VmV38"
# KAGGLE_CONFIG: dataset_sources = []
# KAGGLE_CONFIG: competition_sources = []
# KAGGLE_CONFIG: kernel_sources = []
# KAGGLE_CONFIG: model_sources = []
# KAGGLE_CONFIG: keywords = ["classification"]

In [1]:
from datetime import datetime, timezone
print('executed at:',datetime.now(timezone.utc).isoformat())

executed at: 2026-08-14T02:05:37.131595+00:00


# Notebook Consuming : TPU - Tpu1VmV38 - 128GB

### How the Memory is Structured

Kaggle uses a **TPU v5e-8** pod slice, which is broken down into a multi-core layout:

- **Total TPU Memory: 128 GB**
- **Number of Chips/Cores: 8 v5e chips**
- **Memory Per Core: 16 GB** per chip (8 × 16 GB = 128 GB).

## Importing Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 1. Automatic Kaggle TPU Detection & Strategy Initialization

In [ ]:
try:
    # Detect Kaggle TPU cluster
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print(f"Running on TPU: {tpu.master()}")
    
    # Connect to the cluster and initialize TPU system
    tf.config.experimental_connect_to_cluster(tpu)
    tf.config.experimental.initialize_tpu_system(tpu)
    
    # Define TPU distribution strategy
    strategy = tf.distribute.TPUStrategy(tpu)
    print("✅ Successfully initialized Kaggle TPU strategy.")
except ValueError:
    print("⚠️ TPU not detected. Falling back to default strategy (GPU/CPU).")
    strategy = tf.distribute.get_strategy()

# Adjust batch size for parallel execution across TPU cores
NUM_REPLICAS = strategy.num_replicas_in_sync
BATCH_SIZE = 16 * NUM_REPLICAS
print(f"Number of accelerator replicas in sync: {NUM_REPLICAS}")
print(f"Global Batch Size: {BATCH_SIZE}")

## 2. Data Loading & Preprocessing

In [ ]:
print("\nLoading Breast Cancer dataset from Scikit-Learn...")
data = load_breast_cancer()
X = data.data
y = data.target

# Train-test split (80/20 ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (critical for neural network convergence)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Deep Learning Model Construction (Within TPU Scope)

In [ ]:
# Models MUST be built and compiled inside strategy.scope() for TPU execution
with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

model.summary()

## 4. Model Training

In [ ]:
print("\nStarting Training on TPU...")
history = model.fit(
    X_train_scaled, 
    y_train, 
    epochs=50, 
    batch_size=BATCH_SIZE, 
    validation_split=0.15,
    verbose=1
)

## 5. Model Evaluation & Results Printing

In [ ]:
print("\nEvaluating model on test dataset...")
y_pred_probs = model.predict(X_test_scaled)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

acc = accuracy_score(y_test, y_pred)
print("\n" + "=" * 50)
print(f"Test Accuracy: {acc * 100:.2f}%")
print("=" * 50)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=data.target_names))

cm = confusion_matrix(y_test, y_pred)
print("--- Confusion Matrix ---")
print(cm)

# Plot Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm, 
    annot=True, 
    fmt="d", 
    cmap="Blues", 
    xticklabels=data.target_names, 
    yticklabels=data.target_names
)
plt.title("TPU Model Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

## 6. Save Model to Kaggle Output Path

In [ ]:
kaggle_output_dir = "/kaggle/working" if os.path.exists("/kaggle") else "./output"
os.makedirs(kaggle_output_dir, exist_ok=True)

model_save_path = os.path.join(kaggle_output_dir, "tpu_deep_learning_model.keras")
model.save(model_save_path)

print(f"\n✅ Model successfully saved to Kaggle path: {model_save_path}")